# Chapter 7 · Resources, prompts, sampling and elicitation

What an MCP server can offer besides tools: documents that the application reads when it needs them (resources), ready-made instructions (prompts), texts that the server asks the client's model for (sampling) and questions that the server asks the user (elicitation). We measure whether the documents should be given all at once or on demand, and whether asking the user prevents bookings with invented data.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model and the clinic

The clinic's server is chapter 6's; it is saved to a file so that it can be imported.

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # the versions the book was checked with

In [ ]:
%%writefile clinic_server.py
"""MCP server of Northside Veterinary Clinic: chapter 6's server, with the vaccine and
booking functions outside the server, so that they can be reused.

It can be used in three ways: imported, with a client connected in memory; launched as a
process that talks over stdio; or launched as an HTTP service that several clients
connect to at once.
Usage: python clinic_server.py [data_folder] [--http PORT]
"""
import argparse
import copy
import json
import unicodedata
from datetime import date
from pathlib import Path
from typing import Annotated

from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError
from pydantic import Field

TODAY = date(2026, 10, 20)                  # the date of the clinic's data


# --- The clinic's data -------------------------------------------------------

def load_data(folder="data"):
    """Reads the clinic's files and leaves the schedule as it was at the start."""
    global PETS, OWNERS, HISTORY, VACCINES, APPOINTMENTS0, PHARMACY, INVOICES, VETS
    global TIMES
    read = lambda name: json.loads((Path(folder) / name).read_text(encoding="utf-8"))
    PETS, OWNERS = read("pets.json"), read("owners.json")
    HISTORY, VACCINES = read("history.json"), read("vaccines.json")
    APPOINTMENTS0, PHARMACY = read("appointments.json"), read("pharmacy.json")
    INVOICES, VETS = read("invoices.json"), read("vets.json")
    TIMES = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reset()


def reset():
    """Leaves the clinic as it was at the start: the original schedule, no notices."""
    global APPOINTMENTS, NOTICES
    APPOINTMENTS, NOTICES = copy.deepcopy(APPOINTMENTS0), []


def strip_accents(text):
    """Lowercase and without diacritics, to compare names however they are written."""
    return "".join(c for c in unicodedata.normalize("NFD", str(text).lower())
                   if unicodedata.category(c) != "Mn").strip()


def age(birth_date):
    """Age in whole years on the date of the data."""
    n = date.fromisoformat(birth_date)
    return TODAY.year - n.year - ((TODAY.month, TODAY.day) < (n.month, n.day))


def pet_summary(pid):
    """The main data of a pet, with its owner's name."""
    p = PETS[pid]
    return {"name": p["name"], "species": p["species"], "breed": p["breed"],
            "age_years": age(p["birth_date"]), "current_weight_kg": p["weight_kg"],
            "allergies": p["allergies"] or "none known",
            "owner": OWNERS[p["owner"]]["name"]}


def vet_of(name):
    """The id of the vet with that name (or part of it), or None."""
    for v, n in VETS.items():
        if strip_accents(name) in (strip_accents(v), strip_accents(n)) \
                or strip_accents(name) in strip_accents(n):
            return v
    return None


def pet_of(name, owner=""):
    """The id of the pet with that name (and owner); an error if there are none or
    several."""
    ids = [p for p, d in PETS.items() if strip_accents(name) == strip_accents(d["name"])
           and strip_accents(owner) in strip_accents(OWNERS[d["owner"]]["name"])]
    if not ids:
        raise ToolError(f'There is no pet called "{name}"'
                        + (f" owned by {owner}" if owner else "") + " at the clinic.")
    if len(ids) > 1:
        raise ToolError(f'There are {len(ids)} pets called "{name}"; '
                        "give the owner's name.")
    return ids[0]


def free(date_, vet=""):
    """The free times of a weekday, for one vet or for all."""
    try:
        day = date.fromisoformat(date_)
    except ValueError:
        raise ToolError(f'"{date_}" is not a date; use YYYY-MM-DD.')
    if day.weekday() >= 5:
        raise ToolError(f"{date_} is a weekend day and the clinic is closed.")
    v_id = vet_of(vet) if vet else None
    if vet and v_id is None:
        raise ToolError(f'There is no vet "{vet}". They are: '
                        + ", ".join(VETS.values()) + ".")
    taken = {(a["vet"], a["time"]) for a in APPOINTMENTS
             if a["date"] == date_ and a["status"] != "canceled"}
    return {"date": date_, "free_times": {
        VETS[v]: [t for t in TIMES if (v, t) not in taken]
        for v in ([v_id] if v_id else VETS)}}


def vaccines_of(pet, owner=""):
    """The vaccines of a pet, each with its next date and whether it is overdue."""
    pid = pet_of(pet, owner)
    return {"pet": PETS[pid]["name"], "today": TODAY.isoformat(), "vaccines": [
        {"vaccine": v["vaccine"], "next": v["next"],
         "status": "overdue" if v["next"] < TODAY.isoformat() else "up to date"}
        for v in VACCINES[pid]]}


def book_appointment_data(pet, vet, date, time, reason, owner=""):
    """Books an appointment, if all the data are valid and the time is free."""
    pid, v_id = pet_of(pet, owner), vet_of(vet)
    if v_id is None:
        raise ToolError(f'There is no vet "{vet}".')
    free(date, vet)
    if time not in TIMES:
        raise ToolError(f'"{time}" is not an appointment time.')
    if any(a["date"] == date and a["time"] == time and a["vet"] == v_id
           and a["status"] != "canceled" for a in APPOINTMENTS):
        raise ToolError(f"{VETS[v_id]} already has an appointment on {date} at {time}.")
    appointment = {"id": f"AP-{len(APPOINTMENTS0) + len(APPOINTMENTS) + 1:04d}",
                   "date": date, "time": time, "vet": v_id, "pet": pid,
                   "reason": reason, "status": "scheduled"}
    APPOINTMENTS.append(appointment)
    return {"booked": appointment["id"], "pet": PETS[pid]["name"], "vet": VETS[v_id],
            "date": date, "time": time}


# --- The server --------------------------------------------------------------

Name = Annotated[str, Field(description="Name of the pet.")]
Owner = Annotated[str, Field(
    description="Owner's name; only if several pets have that name.")]
Date = Annotated[str, Field(description="YYYY-MM-DD.")]


def create_server():
    """The clinic's server, with its eight tools."""
    server = MCPServer("clinic", instructions="Pets, vaccines, medical history and "
                       "schedule of Northside Veterinary Clinic.")

    @server.tool()
    def find_pet(name: Name, owner: Owner = "") -> dict:
        """Data of a pet looked up by name: species, age, weight, allergies, owner."""
        return pet_summary(pet_of(name, owner))

    @server.tool()
    def vaccines(pet: Name, owner: Owner = "") -> dict:
        """Vaccines of a pet: the next date of each one and whether it is overdue."""
        return vaccines_of(pet, owner)

    @server.tool()
    def history(pet: Name, last: Annotated[int, Field(
            description="How many visits, from the most recent to the oldest.")] = 3,
                owner: Owner = "") -> dict:
        """The last visits of a pet, starting with the most recent."""
        pid = pet_of(pet, owner)
        return {"pet": PETS[pid]["name"], "visits": [
            {k: v[k] for k in ("date", "reason", "treatment", "weight_kg")}
            for v in reversed(HISTORY[pid][-last:])]}

    @server.tool()
    def free_slots(date: Date, vet: Annotated[str, Field(
            description="Vet's name; empty for all.")] = "") -> dict:
        """Free times in the schedule on a weekday, for one vet or for all."""
        return free(date, vet)

    @server.tool()
    def appointments_of(pet: Name, owner: Owner = "") -> dict:
        """Appointments of a pet (past and upcoming), with their id."""
        pid = pet_of(pet, owner)
        return {"pet": PETS[pid]["name"], "today": TODAY.isoformat(), "appointments": [
            {"id": a["id"], "date": a["date"], "time": a["time"], "reason": a["reason"],
             "vet": VETS[a["vet"]], "status": a["status"]}
            for a in APPOINTMENTS if a["pet"] == pid]}

    @server.tool()
    def book_appointment(pet: Name,
                         vet: Annotated[str, Field(description="Vet's name.")],
                         date: Date, time: Annotated[str, Field(
                             description="HH:MM, on the hour or half past.")],
                         reason: Annotated[str, Field(
                             description="Reason for the appointment.")],
                         owner: Owner = "") -> dict:
        """Books an appointment at a free time."""
        return book_appointment_data(pet, vet, date, time, reason, owner)

    @server.tool()
    def cancel_appointment(appointment: Annotated[str, Field(
            description="Appointment id, like AP-0026 (given by appointments_of).")]
                           ) -> dict:
        """Cancels a scheduled appointment."""
        for a in APPOINTMENTS:
            if a["id"] == appointment:
                if a["status"] != "scheduled":
                    raise ToolError(f'Appointment {appointment} is "{a["status"]}".')
                a["status"] = "canceled"
                return {"canceled": appointment, "date": a["date"], "time": a["time"]}
        raise ToolError(f"Appointment {appointment} does not exist.")

    @server.tool()
    def upcoming_appointments() -> dict:
        """All the scheduled appointments in the schedule, to check what was booked."""
        return {"appointments": [{k: a[k] for k in ("id", "date", "time", "vet", "pet")}
                                 for a in APPOINTMENTS if a["status"] == "scheduled"]}

    return server


if __name__ == "__main__":
    arguments = argparse.ArgumentParser()
    arguments.add_argument("data", nargs="?", default="data")
    arguments.add_argument("--http", type=int,
                           help="port; without it, the server talks over stdio")
    a = arguments.parse_args()
    load_data(a.data)
    server = create_server()
    if a.http:
        server.run(transport="streamable-http", host="127.0.0.1", port=a.http)
    else:
        server.run()

In [ ]:
"""Chapter 7: resources, prompts, sampling and elicitation. What an MCP server offers
besides tools, with two measurements: documents on demand and questions to the user."""
import asyncio
import json
import re
import sys
import urllib.request
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client
from mcp.server.mcpserver import Context
from mcp.server.mcpserver.exceptions import ToolError
from mcp.server.mcpserver.resources import TextResource
from mcp.shared.exceptions import MCPError
from mcp.types import CreateMessageResult, ElicitResult, SamplingMessage, TextContent
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = ("https://raw.githubusercontent.com/HernanDiaz/"
              "ai-tools-mcp-skills/main/en/")
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


import clinic_server as clinic


def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; if not, on the CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def fetch_data():
    """The clinic's files, in the data folder; in Colab, they are downloaded."""
    DATA.mkdir(exist_ok=True)
    for name in ("pets", "owners", "history", "vaccines", "appointments", "pharmacy",
                 "invoices", "vets", "prices"):
        path = DATA / f"{name}.json"
        if not path.exists():
            urllib.request.urlretrieve(REPOSITORY + "data/" + path.name, path)

In [ ]:
tok, model = load(MODEL)
fetch_data()
clinic.load_data(DATA)

In [ ]:
SYSTEM = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in English, in one or two sentences.")


def generate(messages, schemas, max_tokens=400):
    """One answer from the model, and its input and output tokens."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    return (tok.decode(new, skip_special_tokens=True).strip(),
            inputs["input_ids"].shape[1], len(new))


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"],
                          "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


def time_in(answer, hhmm):
    """Does the answer give the time HH:MM, as 16:30 or as 4:30 p.m.?"""
    h, m = int(hhmm[:2]), int(hhmm[3:])
    h12, half = h % 12 or 12, "a" if h < 12 else "p"
    minutes = f":{m:02d}" if m else "(:00)?"
    forms = [rf"(?<![\d:]){h:02d}:{m:02d}(?!\d)", rf"(?<![\d:]){h}:{m:02d}(?!\d)",
             rf"(?<![\d:]){h12}{minutes}\s?{half}\.?\s?m\b"]
    if (h, m) == (12, 0):
        forms.append(r"\bnoon\b")
    return says(answer, *forms)


def without_titles(schema):
    """The schema without the "title" keys that Pydantic adds and the model does not
    need."""
    if isinstance(schema, dict):
        return {k: without_titles(v) for k, v in schema.items() if k != "title"}
    return schema


def compact(text):
    """The JSON of a result on a single line, without the indentation the server
    sends."""
    try:
        return json.dumps(json.loads(text), ensure_ascii=False)
    except ValueError:
        return text

## 2. The server: tools, resources, a prompt and sampling

In [ ]:
DOCUMENTS = {
    "cancellations": ("Appointment cancellation policy",
        "Appointments can be canceled or changed free of charge up to 24 hours before. "
        "If they are canceled less than 24 hours in advance, or the client does not "
        "show up, there is a $15 fee, which is deducted from the next visit if it is "
        "booked in the same month. Emergency cancellations because the pet is ill are "
        "not charged. Cancellations are made by phone or at the front desk; not by "
        "e-mail."),
    "anesthesia": ("Preparing for surgery and anesthesia",
        "Before general anesthesia, dogs and cats must go 12 hours without food; they "
        "can drink water until 2 hours before. Rabbits must not fast: their food is "
        "removed only 1 hour before. On the day of the surgery, the pet arrives at "
        "8:30 a.m. and can be picked up from 5 p.m. Pets older than 8 get a "
        "pre-surgery blood test, which costs $65."),
    "emergencies": ("After-hours emergencies",
        "Outside opening hours, emergencies are handled by the on-call vet at (503) "
        "555-0111. The on-call duty rotates every week among the three vets. An "
        "after-hours emergency has a $60 surcharge on the normal price. Heatstroke, "
        "poisoning and breathing difficulties are always emergencies."),
    "puppies": ("Puppy vaccination plan",
        "Puppies get the first dose of the DHPP vaccine at 8 weeks, the second at 12 "
        "and the third at 16, together with the first rabies vaccine. Until one week "
        "after the third dose they should not go out on the street or mix with other "
        "dogs. Kittens get the FVRCP vaccine at 9 and at 12 weeks."),
    "deworming": ("Deworming schedule",
        "Adult dogs and cats are dewormed internally every 3 months and externally "
        "every month, with a spot-on or a collar. Puppies, every 15 days until 3 "
        "months of age. Before a vaccine, the pet must have been dewormed internally "
        "in the previous 2 weeks; if not, it is dewormed at the visit and the vaccine "
        "is given 10 days later."),
    "payments": ("Payments and invoices",
        "Visits are paid at the end, by card, cash or Apple Pay. Surgeries are paid "
        "half when booking and the rest when picking up the pet. Unpaid invoices get a "
        "reminder at 30 days; at 60 days no new appointments are given, except "
        "emergencies, until they are paid. Clients with more than three pets get a 10% "
        "discount on vaccines."),
    "neuter": ("Care after a neuter",
        "After a neuter, the pet wears a cone for 10 days so that it does not lick the "
        "wound. The stitch check is at 10 days and is free. For one week, short walks "
        "and no running. If the wound opens, bleeds or swells, call the clinic the "
        "same day."),
    "heatstroke": ("Heatstroke",
        "In case of heatstroke, take the pet to the shade, wet it with cool water, "
        "never ice-cold, and come to the clinic immediately. Flat-faced breeds, such "
        "as the French bulldog or the Persian, are at higher risk. In summer, walks "
        "should be before 10 a.m. or after 8 p.m."),
}


MORE_DOCUMENTS = {
    "microchip": ("Microchip identification",
        "All dogs, cats and ferrets should be microchipped from three months of age. "
        "The clinic implants the microchip during the visit, without anesthesia, for "
        "$35, and registers the pet in a national pet microchip registry on the same "
        "day. If the owner moves or changes phone number, they must tell the clinic so "
        "that the registry is updated."),
    "travel": ("Travel and the health certificate",
        "To travel to the European Union, the pet needs a microchip, an EU health "
        "certificate and the rabies vaccine given at least 21 days before the trip. "
        "The certificate costs $20 and is issued at the visit. Some countries, such as "
        "Ireland or Finland, also require a tapeworm treatment between 1 and 5 days "
        "before arrival."),
    "boarding": ("Boarding and daycare",
        "The clinic has no boarding of its own, but works with Killingsworth Dog "
        "Lodge. To stay there, the dog must be up to date on DHPP, rabies and the "
        "kennel cough (Bordetella) vaccine, given at least 15 days before. The clinic "
        "issues the certificate free of charge to its clients."),
    "summer": ("Summer hours",
        "In July and August, the clinic is open Monday to Friday from 9 a.m. to 2 "
        "p.m., and in the afternoon only on Tuesdays and Thursdays, from 5 p.m. to 8 "
        "p.m. Emergencies are still handled by the on-call vet. Scheduled surgeries "
        "are concentrated in June and September."),
    "adoption": ("Checkup for adopted pets",
        "Pets adopted from animal shelters in Oregon get their first checkup free if "
        "it is booked within the first month. At that visit the vet reviews their "
        "general condition, checks the microchip and plans the vaccines and deworming "
        "that are missing."),
    "end_of_life": ("End of life",
        "When a pet is in its last days, the vet explains to the family the options "
        "for palliative care and, if they so decide, euthanasia, which is always done "
        "with prior sedation. The clinic arranges individual or communal cremation "
        "with an outside service and does not charge for the farewell visit."),
    "insert_meloxicam": ("Package insert: meloxicam",
        "Anti-inflammatory for dogs and cats, as an oral suspension of 1.5 mg/ml. "
        "Dose: 0.1 mg per kg once a day, with food. Do not give to dehydrated pets or "
        "pets with kidney problems. If vomiting or black stools appear, stop and call "
        "the clinic."),
    "insert_amoxicillin": ("Package insert: amoxicillin",
        "Antibiotic for dogs, cats and rabbits, in 50 mg tablets. Dose: 10 mg per kg "
        "twice a day for 7 days, even if the pet gets better sooner. Do not use in "
        "pets allergic to penicillins. In rabbits, watch the stools."),
    "insert_doxycycline": ("Package insert: doxycycline",
        "Antibiotic for dogs and cats, in 100 mg tablets. Dose: 5 mg per kg once a "
        "day. Give with food and water, never dry, especially to cats. Do not use in "
        "growing puppies."),
    "insert_maropitant": ("Package insert: maropitant",
        "Anti-vomiting drug for dogs only, in 16 mg tablets. Dose: 2 mg per kg once a "
        "day, for at most 5 days in a row. For motion sickness on trips, give it one "
        "hour before leaving."),
    "insert_praziquantel": ("Package insert: praziquantel",
        "Internal antiparasitic for dogs and cats, in 50 mg tablets. Dose: 5 mg per kg "
        "in a single dose. It is the tapeworm treatment that some countries require "
        "for travel."),
    "insert_enrofloxacin": ("Package insert: enrofloxacin",
        "Antibiotic for dogs, cats and rabbits, in 15 mg tablets. Dose: 5 mg per kg "
        "once a day. Do not use in dogs under one year or in cats under eight weeks."),
    "insert_gabapentin": ("Package insert: gabapentin",
        "For chronic pain and for anxiety before a visit, in dogs and cats, in 100 mg "
        "capsules. Dose: 10 mg per kg. It can cause drowsiness; for visits, give it "
        "two hours before."),
}

In [ ]:
Name = Annotated[str, Field(description="Name of the pet.")]


class MissingTime(BaseModel):
    """What the tool asks when the time is missing."""
    time: str = Field(description="Time of the appointment, HH:MM.")


class MissingVet(BaseModel):
    """What the tool asks when the vet is missing."""
    vet: str = Field(description="Vet's name.")


class MissingDate(BaseModel):
    """What the tool asks when the date is missing."""
    date: str = Field(description="Day of the appointment, YYYY-MM-DD.")


class MissingReason(BaseModel):
    """What the tool asks when the reason is missing."""
    reason: str = Field(description="Reason for the appointment.")


MISSING = {"time": MissingTime, "vet": MissingVet, "date": MissingDate,
           "reason": MissingReason}


class Confirmation(BaseModel):
    """The whole appointment, which the user confirms or corrects."""
    vet: str = Field(description="Vet's name.")
    date: str = Field(description="Day of the appointment, YYYY-MM-DD.")
    time: str = Field(description="Time of the appointment, HH:MM.")
    reason: str = Field(description="Reason for the appointment.")


def create_server(ask="no", documents=DOCUMENTS):
    """Chapter 6's clinic server, with the documents as resources, a prompt and a tool
    that writes with the client's model. ask says what book_appointment does: "no"
    books whatever it is asked; "missing" accepts empty data and asks the user for the
    missing ones; "confirm" shows the user the appointment before booking it, so that
    they correct whatever is not what they asked for (the last two, with
    elicitation)."""
    server = clinic.create_server()
    server.remove_tool("book_appointment")
    book = clinic.book_appointment_data

    if ask == "missing":
        @server.tool()
        async def book_appointment(pet: Name, ctx: Context, vet: str = "",
                                   date: str = "", time: str = "",
                                   reason: str = "") -> dict:
            """Books an appointment. Any data left empty is asked of the user."""
            data = {"vet": vet, "date": date, "time": time, "reason": reason}
            for field, value in data.items():
                if not value:
                    r = await ctx.elicit(f"To book {pet}'s appointment, what {field}?",
                                         MISSING[field])
                    if r.action != "accept":
                        raise ToolError(f"Booking canceled: the {field} is missing.")
                    data[field] = getattr(r.data, field)
            return book(pet, **data)
    elif ask == "confirm":
        @server.tool()
        async def book_appointment(pet: Name, vet: str, date: str, time: str,
                                   reason: str, ctx: Context) -> dict:
            """Books an appointment the user confirms. Dates YYYY-MM-DD, times HH:MM."""
            r = await ctx.elicit(f"I am going to book for {pet}: {vet}, {date} at "
                                 f'{time}, reason "{reason}". Correct anything that '
                                 "is not right.", Confirmation)
            if r.action != "accept":
                raise ToolError("The user has not confirmed the booking.")
            proposal = {"vet": vet, "date": date, "time": time, "reason": reason}
            changes = {c: f"{proposal[c]} → {v}"
                       for c, v in r.data.model_dump().items() if v != proposal[c]}
            return {**book(pet, **r.data.model_dump()),
                    "changed_by_user": changes or "nothing"}
    else:
        @server.tool()
        def book_appointment(pet: Name, vet: str, date: str, time: str,
                             reason: str) -> dict:
            """Books an appointment. Dates YYYY-MM-DD; times HH:MM, :00 or :30."""
            return book(pet, vet=vet, date=date, time=time, reason=reason)

    for key, (title, text) in documents.items():
        server.add_resource(TextResource(uri=f"clinic://documents/{key}", name=key,
                                         description=title, text=text))

    @server.prompt()
    def vaccine_notice(pet: str) -> str:
        """Asks the model for a notice of overdue vaccines for a pet's owner."""
        return (f"Check {pet}'s vaccines with the vaccines tool and write, in two "
                "sentences and in a friendly tone, a notice for the owner with the "
                "ones that are overdue.")

    @server.tool()
    async def draft_notice(pet: Name, ctx: Context) -> dict:
        """Writes the text of a notice of overdue vaccines for a pet's owner."""
        data = clinic.vaccines_of(pet)
        overdue = [v["vaccine"] for v in data["vaccines"] if v["status"] == "overdue"]
        request = (f"Write in one sentence a friendly notice for {pet}'s owner: these "
                   f"vaccines are overdue: {', '.join(overdue) or 'none'}.")
        r = await ctx.session.create_message(
            messages=[SamplingMessage(role="user",
                                      content=TextContent(type="text", text=request))],
            max_tokens=80)
        return {"pet": pet, "overdue": overdue, "notice": r.content.text}

    return server

## 3. The host

It gives the documents to the model in two ways, answers the server's questions with a simulated user and serves the sampling requests with the local model.

In [ ]:
class Host:
    """Chapter 4's compact host, for one server, with three additions: it can give the
    model the documents in two ways (all in the system message, or a tool to read them
    when needed), it answers the server's questions with a simulated user, and it
    serves the sampling requests with the local model."""

    def __init__(self, server, documents="none", user=None):
        """Keeps the server, how to give the documents and the simulated user."""
        self.server, self.documents, self.user = server, documents, user
        self.questions = []

    async def __aenter__(self):
        """Connects, lists the tools and prepares the documents."""
        self.client = Client(self.server, mode="legacy",
                             elicitation_callback=self.answer,
                             sampling_callback=self.sample)
        await self.client.__aenter__()
        self.schemas = [{"type": "function", "function": {
            "name": t.name, "description": t.description,
            "parameters": without_titles(t.input_schema)}}
            for t in (await self.client.list_tools()).tools if t.name != "draft_notice"]
        resources = (await self.client.list_resources()).resources
        self.extra = ""
        if self.documents == "all":
            texts = [(r.description,
                      (await self.client.read_resource(r.uri)).contents[0].text)
                     for r in resources if str(r.uri).startswith("clinic://documents/")]
            self.extra = "\n\nDocuments of the clinic:\n" + "\n".join(
                f"# {t}\n{x}" for t, x in texts)
        elif self.documents == "on demand":
            uris = [str(r.uri) for r in resources
                    if str(r.uri).startswith("clinic://documents/")]
            self.schemas.append({"type": "function", "function": {
                "name": "read_document",
                "description": "Reads a document of the clinic. Documents: "
                + "; ".join(f"{str(r.uri)} ({r.description})"
                            for r in resources if str(r.uri) in uris) + ".",
                "parameters": {"type": "object", "required": ["uri"], "properties": {
                    "uri": {"type": "string", "enum": uris}}}}})
        return self

    async def __aexit__(self, *error):
        """Closes the connection."""
        await self.client.__aexit__(*error)

    async def answer(self, context, params):
        """Elicitation: the simulated user, who knows which appointment they want,
        answers whatever the server asks with the data of that appointment."""
        fields = list(params.requested_schema["properties"])
        self.questions.append(params.message)
        if self.user and all(f in self.user for f in fields):
            return ElicitResult(action="accept",
                                content={f: self.user[f] for f in fields})
        return ElicitResult(action="decline")

    async def sample(self, context, params):
        """Sampling: the server asks for a text and the host's model writes it."""
        messages = [{"role": m.role, "content": m.content.text}
                    for m in params.messages]
        text = generate(messages, [], max_tokens=params.max_tokens)[0]
        return CreateMessageResult(role="assistant",
                                   content=TextContent(type="text", text=text),
                                   model=GPU_MODEL)

    async def call(self, name, arguments):
        """Runs a call of the model: a document or a tool of the server."""
        if name == "read_document":
            try:
                r = await self.client.read_resource(arguments.get("uri", ""))
            except MCPError as e:  # a document that does not exist: the model reads why
                return json.dumps({"error": str(e)}, ensure_ascii=False)
            return r.contents[0].text
        r = await self.client.call_tool(name, arguments)
        return compact("\n".join(b.text for b in r.content if b.type == "text"))


async def mcp_agent(task, host, max_steps=8, show=False):
    """Chapter 4's loop, with the host's extra text in the system message."""
    messages = [{"role": "system", "content": SYSTEM + host.extra},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_input, n_output = generate(messages, host.schemas)
        calls = read_calls(answer)[:1]
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_input, "calls": []})
        if not calls:
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            text = await host.call(call["name"], call["arguments"])
            trace[-1]["calls"].append({**call, "result": text})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                arguments = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({arguments})")
                print(f"     → {text[:150]}")
    return None, trace

In [ ]:
async def demonstration():
    async with Host(create_server()) as h:
        resources = (await h.client.list_resources()).resources
        print([str(r.uri) for r in resources])
        prompt = await h.client.get_prompt("vaccine_notice", {"pet": "Toby"})
        print("Prompt:", prompt.messages[0].content.text)
        r = await h.client.call_tool("draft_notice", {"pet": "Toby"})
        print("Sampling:", r.content[0].text)

await demonstration()

## 4. The documents: all or on demand

In [ ]:
QUESTIONS = [
    ("late_cancel", "A client is canceling today the appointment she has tomorrow. "
     "Do we charge her anything?", lambda r: says(r, r"\b15\b")),
    ("cat_fasting", "How many hours does a cat have to go without food before "
     "anesthesia?", lambda r: says(r, r"\b12\b")),
    ("rabbit_fasting", "A rabbit is having surgery tomorrow. How many hours of fasting "
     "does it need?",
     lambda r: says(r, r"\b1[- ]hour", r"\bone[- ]hour", r"\ban hour")),
    ("surcharge", "What is the surcharge for an after-hours emergency?",
     lambda r: says(r, r"\b60\b")),
    ("puppies", "At how many weeks does a puppy get the second dose of DHPP?",
     lambda r: says(r, r"\b12\b", "twelve")),
    ("deworming", "How often is an adult dog dewormed internally?",
     lambda r: says(r, r"\b3[- ]months?", r"three[- ]months?")),
    ("surgery_payment", "How is a surgery paid for?", lambda r: says(r, r"\bhalf\b")),
    ("cone", "How many days does a neutered dog have to wear the cone?",
     lambda r: says(r, r"\b10\b", r"\bten\b")),
]


COLLECTIONS = {"8 documents": DOCUMENTS,
               "21 documents": {**DOCUMENTS, **MORE_DOCUMENTS}}


async def measure_documents():
    """The eight questions with no documents, and with 8 and 21, all or on demand."""
    results = {}
    variants = [("none", "8 documents")] + [(d, c) for c in COLLECTIONS
                                            for d in ("all", "on demand")]
    for documents, collection in variants:
        rows = []
        async with Host(create_server(documents=COLLECTIONS[collection]),
                        documents) as host:
            for key, question, check in QUESTIONS:
                answer, trace = await mcp_agent(question, host)
                rows.append({"task": key, "ok": bool(check(answer)), "answer": answer,
                             "input_tokens": sum(p["input_tokens"] for p in trace),
                             "calls": [(c["name"], c["arguments"])
                                       for p in trace for c in p["calls"]]})
        name = "none" if documents == "none" else f"{documents}, {collection}"
        results[name] = {"ok": sum(r["ok"] for r in rows),
                         "input_tokens": sum(r["input_tokens"] for r in rows),
                         "rows": rows}
        print(f"documents {name:24s}", results[name]["ok"], "of", len(QUESTIONS),
              results[name]["input_tokens"], "tokens", flush=True)
    return results

In [ ]:
documents = await measure_documents()

## 5. Bookings with a piece of data missing

In [ ]:
INCOMPLETE = [
    ("no_time", "Book a checkup for Lola with Dr. Anna Sutton on Monday the 26th.",
     {"pet": "Lola", "vet": "Anna Sutton", "date": "2026-10-26", "time": "11:00",
      "reason": "checkup"}),
    ("no_vet", "Book a vaccination for Max on Monday the 26th at noon.",
     {"pet": "Max", "vet": "Lucy Fenton", "date": "2026-10-26", "time": "12:00",
      "reason": "vaccination"}),
    ("no_date", "Book a checkup for Toby with Dr. Paul Meyer at 10 a.m.",
     {"pet": "Toby", "vet": "Paul Meyer", "date": "2026-10-27", "time": "10:00",
      "reason": "checkup"}),
    ("no_reason", "Book an appointment for Rocky with Dr. Lucy Fenton on Tuesday the "
     "27th at 4:30 p.m.",
     {"pet": "Rocky", "vet": "Lucy Fenton", "date": "2026-10-27", "time": "16:30",
      "reason": "weight check"}),
    ("afternoon", "Book a checkup for Coco with Dr. Anna Sutton on Monday the 26th in "
     "the afternoon.",
     {"pet": "Coco", "vet": "Anna Sutton", "date": "2026-10-26", "time": "17:30",
      "reason": "checkup"}),
    ("complete", "Book a consultation for Kira with Dr. Paul Meyer on Monday the 26th "
     "at 9:30 a.m.",
     {"pet": "Kira", "vet": "Paul Meyer", "date": "2026-10-26", "time": "09:30",
      "reason": "consultation"}),
]


def new_appointments():
    """The appointments that were not in the original schedule."""
    original = {x["id"] for x in clinic.APPOINTMENTS0}
    return [a for a in clinic.APPOINTMENTS if a["id"] not in original]


def is_the_appointment(appointment, wanted):
    """Is it the appointment the user wanted? The reason only has to contain it."""
    return (clinic.PETS[appointment["pet"]]["name"] == wanted["pet"]
            and appointment["vet"] == clinic.vet_of(wanted["vet"])
            and (appointment["date"], appointment["time"]) == (wanted["date"],
                                                               wanted["time"])
            and clinic.strip_accents(wanted["reason"])
            in clinic.strip_accents(appointment["reason"]))


async def measure_questions():
    """The six bookings with the three versions of book_appointment."""
    results = {}
    for ask in ("no", "missing", "confirm"):
        rows = []
        for key, task, wanted in INCOMPLETE:
            clinic.reset()
            async with Host(create_server(ask), user=wanted) as host:
                answer, trace = await mcp_agent(task, host)
                questions = host.questions
            booked = new_appointments()
            day = int(booked[0]["date"][-2:]) if len(booked) == 1 else 0
            rows.append({"task": key, "ok": len(booked) == 1
                         and is_the_appointment(booked[0], wanted),
                         "invented": sum(not is_the_appointment(a, wanted)
                                         for a in booked),
                         "answer_matches": len(booked) == 1
                         and time_in(answer, booked[0]["time"])
                         and says(answer, rf"\b{day}(st|nd|rd|th)?\b"),
                         "answer": answer, "questions": questions,
                         "new_appointments": booked,
                         "calls": [(c["name"], c["arguments"])
                                   for p in trace for c in p["calls"]]})
        results[ask] = {"ok": sum(r["ok"] for r in rows),
                        "appointments_with_invented_data": sum(r["invented"]
                                                               for r in rows),
                        "matching_answers": sum(r["answer_matches"] for r in rows),
                        "questions": sum(len(r["questions"]) for r in rows),
                        "rows": rows}
        print(f"bookings {ask:10s}",
              {k: v for k, v in results[ask].items() if k != "rows"}, flush=True)
        print("    " + " ".join(("+" if r["ok"] else "-") + r["task"] for r in rows),
              flush=True)
    return results

In [ ]:
bookings = await measure_questions()

## Exercises

1. Explain in your own words who decides to use each of the four things in the chapter: a tool, a resource, a prompt and a sampling request.

2. Read the 21 documents with `read_resource` and count their tokens. From how many documents of that size on does reading them on demand pay off?

3. Change the description of `read_document` so that it gives only the addresses, without the titles. Does the model find the document it needs?

4. Write two questions whose answer is in two documents at once. Does the model read both?

5. In the booking without a time, with the version that asks for what is missing, the model chose a time without asking even though the tool said to leave it empty. Try saying it in the system message too. Does it leave the time empty now?

6. Make the simulated user reject the confirmation of a booking. What does the agent answer?

7. Write a server prompt for a vet's weekly summary and use it as the agent's first message. Which tools does it call?

8. `draft_notice` asks the host's model for the text. Change the host so that, before answering, it shows the request to the user and lets them cancel it. Why would a host want to do this?

9. How many input tokens does each confirmation question add to the model? Count also how many questions each version asks the user in the six bookings. Is it worth always asking, or only when something is missing?

10. **Project.** Add two or three documents as resources to your project from chapter 4, and add a confirmation with elicitation to one of its tools that act. Measure the accuracy and the tokens with and without each change.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [MCP specification](https://modelcontextprotocol.io/specification): resources, prompts, sampling and elicitation. [Python SDK](https://github.com/modelcontextprotocol/python-sdk). Northside Veterinary Clinic, its data and its documents are fictional. Original code and experiments.